# Oil & Gas (Field Operations) — Pre-Filled Configuration
### Ready-to-Run Config for the Oil & Gas Use Case

This is a **pre-configured** version of `00_Industry_Config` specifically for the
**Field Engineer Documentation Burden** demo. All table lists, artifact names,
and data paths are hardcoded from the existing 23-CSV oil & gas dataset.

**Use this instead of `00_Industry_Config` if you want to skip auto-discovery and run
directly with the known oil & gas tables.**

---

### Data Summary
- **6 Dimensions** — Field Engineers, Well Sites, Facilities, Report Types, Equipment, Regulatory Bodies
- **6 Batch Facts** — Daily Drilling Reports, Permit to Work, Field Wellness, Report Quality, Operator Satisfaction, Production Performance
- **6 Event Facts** → Eventhouse — SCADA Interactions, HSE Inspections, Well Integrity Events, Tour Handoffs, Production Alerts, Field Location
- **5 Streams** → Eventhouse — SCADA Alarms, Well Telemetry, PTW Status, HSE Events, Environmental Monitoring

### Ontology
- **6 Entity Types:** FieldEngineer, WellSite, Facility, ReportType, Equipment, RegulatoryBody
- **5 Relationship Types:** assigned_to_well, operates_facility, files_report, maintains_equipment, regulated_by
- **6 Contextualizations:** FieldDocEvent, SCADAInteraction, TourHandoff, WellControlEvent, PermitScan, ProductionAlert

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# INDUSTRY SETTING
# ════════════════════════════════════════════════════════════════════════

INDUSTRY       = "oil_and_gas"
INDUSTRY_LABEL = "Oil & Gas"

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# FABRIC ARTIFACT NAMES
# ════════════════════════════════════════════════════════════════════════
# Update these to match your Fabric workspace artifact names.

LAKEHOUSE_NAME      = "OilGasLakehouse"
WAREHOUSE_NAME      = "OilGas_Datawarehouse"
EVENTHOUSE_NAME     = "oilgas_rt_store"
ONTOLOGY_NAME       = "OilGasOpsOntology"
DATA_AGENT_NAME     = "OilGasQA"
OPS_AGENT_NAME      = "OilGasDocBurden"
SEMANTIC_MODEL_NAME = "OilGas_Ops_Model"

print(f"Lakehouse:      {LAKEHOUSE_NAME}")
print(f"Warehouse:      {WAREHOUSE_NAME}")
print(f"Eventhouse:     {EVENTHOUSE_NAME}")
print(f"Ontology:       {ONTOLOGY_NAME}")
print(f"Data Agent:     {DATA_AGENT_NAME}")
print(f"Ops Agent:      {OPS_AGENT_NAME}")
print(f"Semantic Model: {SEMANTIC_MODEL_NAME}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# DATA PATHS & EVENTHOUSE CONNECTION
# ════════════════════════════════════════════════════════════════════════

# CSV files location in Lakehouse Files area
CSV_BASE_PATH = "/lakehouse/default/Files/oil_gas_field_operations/data"

# Schemas
LAKEHOUSE_SCHEMA = "dbo"
WAREHOUSE_SCHEMA = "dbo"

# ── Eventhouse Connection ───────────────────────────────────────────────
# Fill these in from your Fabric workspace
EVENTHOUSE_CLUSTER_URI = ""   # e.g. "https://trd-xxxxx.z5.kusto.fabric.microsoft.com"
EVENTHOUSE_DATABASE    = ""   # e.g. "oilgas_rt_store"

# Ontology package path
ONTOLOGY_PACKAGE_PATH = "/lakehouse/default/Files/oilgas_ops_ontology_package.iq"

print(f"CSV source:     {CSV_BASE_PATH}")
print(f"LH schema:      {LAKEHOUSE_SCHEMA}")
print(f"WH schema:      {WAREHOUSE_SCHEMA}")
print(f"Eventhouse:     {EVENTHOUSE_CLUSTER_URI or '(fill in your cluster URI)'}")
print(f"Ontology pkg:   {ONTOLOGY_PACKAGE_PATH}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# OIL & GAS TABLE DEFINITIONS (pre-filled, no auto-discovery needed)
# ════════════════════════════════════════════════════════════════════════

import os, json

# ── 6 Dimension Tables → Lakehouse + Warehouse ──────────────────────────
DIM_TABLES = [
    "dim_field_engineers",
    "dim_well_sites",
    "dim_facilities",
    "dim_report_types",
    "dim_equipment",
    "dim_regulatory_bodies",
]

# ── 6 Batch Fact Tables → Lakehouse + Warehouse ─────────────────────────
FACT_TABLES_BATCH = [
    "fact_daily_drilling_reports",
    "fact_permit_to_work",
    "fact_field_wellness",
    "fact_report_quality",
    "fact_operator_satisfaction",
    "fact_production_performance",
]

# ── 6 Event Fact Tables → Eventhouse (KQL) ──────────────────────────────
FACT_TABLES_EVENT = [
    "fact_scada_interactions",
    "fact_hse_inspections",
    "fact_well_integrity_events",
    "fact_tour_handoffs",
    "fact_production_alerts",
    "fact_field_location",
]

# ── 5 Streaming Tables → Eventhouse (KQL) ───────────────────────────────
STREAM_TABLES = [
    "stream_scada_alarms",
    "stream_well_telemetry",
    "stream_ptw_status",
    "stream_hse_events",
    "stream_environmental_monitoring",
]

# ── Combined Target Lists ───────────────────────────────────────────────
LAKEHOUSE_TABLES  = DIM_TABLES + FACT_TABLES_BATCH
WAREHOUSE_TABLES  = DIM_TABLES + FACT_TABLES_BATCH + FACT_TABLES_EVENT
EVENTHOUSE_TABLES = FACT_TABLES_EVENT + STREAM_TABLES

# ── KQL Table Name Mapping (CSV name → KQL table name) ─────────────────
# Event facts strip the 'fact_' prefix; streams strip 'stream_' prefix
EVENTHOUSE_KQL_NAMES = {
    "fact_scada_interactions":           "scada_interactions",
    "fact_hse_inspections":              "hse_inspections",
    "fact_well_integrity_events":        "well_integrity_events",
    "fact_tour_handoffs":                "tour_handoffs",
    "fact_production_alerts":            "production_alerts",
    "fact_field_location":               "field_location",
    "stream_scada_alarms":               "scada_alarms",
    "stream_well_telemetry":             "well_telemetry",
    "stream_ptw_status":                 "ptw_status",
    "stream_hse_events":                 "hse_events",
    "stream_environmental_monitoring":   "environmental_monitoring",
}

print(f"{'='*60}")
print(f"OIL & GAS TABLE INVENTORY")
print(f"{'='*60}")
print(f"\nDimension tables ({len(DIM_TABLES)}):")
for t in DIM_TABLES: print(f"  • {t}")
print(f"\nFact tables — Batch ({len(FACT_TABLES_BATCH)}):")
for t in FACT_TABLES_BATCH: print(f"  • {t}")
print(f"\nFact tables — Event/Eventhouse ({len(FACT_TABLES_EVENT)}):")
for t in FACT_TABLES_EVENT: print(f"  • {t}  →  {EVENTHOUSE_KQL_NAMES[t]}")
print(f"\nStreaming tables — Eventhouse ({len(STREAM_TABLES)}):")
for t in STREAM_TABLES: print(f"  • {t}  →  {EVENTHOUSE_KQL_NAMES[t]}")
print(f"\n{'─'*60}")
print(f"Lakehouse target:  {len(LAKEHOUSE_TABLES)} tables (12 Delta tables)")
print(f"Warehouse target:  {len(WAREHOUSE_TABLES)} tables (18 SQL tables)")
print(f"Eventhouse target: {len(EVENTHOUSE_TABLES)} tables (11 KQL tables)")
print(f"Total:             23 CSV files")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# EXPECTED ROW COUNTS (for validation in downstream notebooks)
# ════════════════════════════════════════════════════════════════════════

EXPECTED_ROW_COUNTS = {
    # Dimensions (117 total)
    "dim_field_engineers":               25,
    "dim_well_sites":                    20,
    "dim_facilities":                     8,
    "dim_report_types":                  18,
    "dim_equipment":                     40,
    "dim_regulatory_bodies":              6,
    # Batch Facts (620 total)
    "fact_daily_drilling_reports":       200,
    "fact_permit_to_work":              120,
    "fact_field_wellness":               25,
    "fact_report_quality":              200,
    "fact_operator_satisfaction":         15,
    "fact_production_performance":        60,
    # Event Facts (1190 total)
    "fact_scada_interactions":          400,
    "fact_hse_inspections":             100,
    "fact_well_integrity_events":        60,
    "fact_tour_handoffs":                50,
    "fact_production_alerts":            80,
    "fact_field_location":              500,
    # Streams (400 total)
    "stream_scada_alarms":               80,
    "stream_well_telemetry":             80,
    "stream_ptw_status":                 80,
    "stream_hse_events":                 80,
    "stream_environmental_monitoring":   80,
}

total_lakehouse = sum(EXPECTED_ROW_COUNTS[t] for t in LAKEHOUSE_TABLES)
total_eventhouse = sum(EXPECTED_ROW_COUNTS[t] for t in EVENTHOUSE_TABLES)
print(f"Expected Lakehouse rows:  {total_lakehouse:,}")
print(f"Expected Eventhouse rows: {total_eventhouse:,}")
print(f"Expected total rows:      {sum(EXPECTED_ROW_COUNTS.values()):,}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# SCHEMA INSPECTION HELPER
# ════════════════════════════════════════════════════════════════════════

def preview_table(table_name, base_path=CSV_BASE_PATH, rows=5):
    """Read a CSV and display schema + sample rows."""
    path = f"{base_path}/{table_name}.csv"
    df = spark.read.option("header", True).option("inferSchema", True).csv(path)
    print(f"\n{'─'*60}")
    print(f"Table: {table_name}  |  {df.count()} rows  |  {len(df.columns)} columns")
    print(f"{'─'*60}")
    df.printSchema()
    df.show(rows, truncate=False)
    return df

# Uncomment to preview specific tables:
# preview_table("dim_field_engineers")
# preview_table("fact_daily_drilling_reports")
# preview_table("stream_well_telemetry")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# CONFIG SUMMARY (exported to downstream notebooks via %run)
# ════════════════════════════════════════════════════════════════════════

CONFIG = {
    "industry":             INDUSTRY,
    "industry_label":       INDUSTRY_LABEL,
    "lakehouse_name":       LAKEHOUSE_NAME,
    "warehouse_name":       WAREHOUSE_NAME,
    "eventhouse_name":      EVENTHOUSE_NAME,
    "eventhouse_uri":       EVENTHOUSE_CLUSTER_URI,
    "eventhouse_db":        EVENTHOUSE_DATABASE,
    "ontology_name":        ONTOLOGY_NAME,
    "ontology_package":     ONTOLOGY_PACKAGE_PATH,
    "data_agent_name":      DATA_AGENT_NAME,
    "ops_agent_name":       OPS_AGENT_NAME,
    "semantic_model_name":  SEMANTIC_MODEL_NAME,
    "csv_base_path":        CSV_BASE_PATH,
    "lakehouse_schema":     LAKEHOUSE_SCHEMA,
    "warehouse_schema":     WAREHOUSE_SCHEMA,
    "dim_tables":           DIM_TABLES,
    "fact_tables_batch":    FACT_TABLES_BATCH,
    "fact_tables_event":    FACT_TABLES_EVENT,
    "stream_tables":        STREAM_TABLES,
    "lakehouse_tables":     LAKEHOUSE_TABLES,
    "warehouse_tables":     WAREHOUSE_TABLES,
    "eventhouse_tables":    EVENTHOUSE_TABLES,
}

print(f"\n{'═'*60}")
print(f"✅ OIL & GAS CONFIG READY")
print(f"{'═'*60}")
print(json.dumps({k: v if not isinstance(v, list) else f"{len(v)} tables" for k, v in CONFIG.items()}, indent=2))
print(f"\nThis config is imported by downstream notebooks via:")
print(f"  %run ./OilAndGas_Config")